## Prompt Stability Analysis

**Why this matters:** Törnberg (2024) identifies prompt sensitivity as a key validity threat for LLM annotation. If small rewording shifts scores significantly, the annotations are not stable and cannot be treated as reliable.

We test this on a 30-tweet random subset using two prompt variants:
- **Prompt A (canonical):** The full codebook as defined above
- **Prompt B (rephrased):** Minor rewording of factor labels (same meaning, different surface form)

We then compute Spearman ρ between the two sets to assess stability.

> Want to report the stability results as evidence of prompt robustness.

In [ ]:
import sys
import os
import time
import json
import re
import math
from datetime import datetime

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
from together import Together 



In [ ]:

# API Keys
load_dotenv() 

# Load keys 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")

# Create clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)

anthropic_client = anthropic.Anthropic(
    api_key=ANTHROPIC_API_KEY
) if ANTHROPIC_API_KEY else None

together_client = Together(
    api_key=TOGETHER_API_KEY
) if TOGETHER_API_KEY else None

MODELS = {
    1: {
        "llm_name_version": "gpt-4o-2024-08-06",
        "provider":         "openai",
        "display_name":     "GPT-4o",
        "rationale":        "Benchmark; used in Jia et al. 2024, Törnberg 2023/2025, Gilardi 2023"
    },
    2: {
        "llm_name_version": "claude-haiku-4-5-20251001",
        "provider":         "anthropic",
        "display_name":     "claude-haiku-4-5",
        "rationale":        "Efficient model for annotation tasks"
    },
    3: {
        "llm_name_version": "meta-llama/Llama-3.2-3B-Instruct-Turbo",
        "provider":         "together",
        "display_name":     "Llama-3.2-3B",
        "rationale":        "Small open-weight baseline; genuine size contrast to Llama-3.3-70B"
    },
}

In [ ]:

# Output Format Specification
OUTPUT_FORMAT = """
Respond ONLY with a JSON object in exactly this format (no extra fields, no markdown):
{
  "V1": {"score": <1|2|3>, "V1_reason": "<brief reason citing tweet language>"},
  "V2": {"score": <1|2|3>, "V2_reason": "<brief reason citing tweet language>"},
  "V3": {"score": <1|2|3>, "V3_reason": "<brief reason citing tweet language>"},
  "V4": {"score": <1|2|3>, "V4_reason": "<brief reason citing tweet language>"},
  "V5": {"score": <1|2|3>, "V5_reason": "<brief reason citing tweet language>"},
  "V6": {"score": <1|2|3>, "V6_reason": "<brief reason citing tweet language>"},
  "V7": {"score": <1|2|3>, "V7_reason": "<brief reason citing tweet language>"},
  "V8": {"score": <1|2|3>, "V8_reason": "<brief reason citing tweet language>"}
}
"""

In [ ]:
# Alternate prompt variant for stability testing
# Same definitions, slightly different phrasing — tests whether results are
# sensitive to surface-level wording of the prompt.

CODEBOOK_ALT = """
ANNOTATION GUIDE: Measuring Anti-Democratic Content in Social Media Posts

V1: HOSTILITY TOWARD THE OPPOSING PARTY
Meaning: Negative attitudes directed at members of the other political party.
Check for:
  A: Derogatory labels targeting the other party
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present

V2: BACKING ANTI-DEMOCRATIC BEHAVIOUR
Meaning: Tolerating or endorsing actions that undermine democratic norms.
Check for:
  A: Endorsement of anti-democratic behaviour
  B1: Derogatory labels  |  B2: Strong language or hyperbole
Scale:
  1 = None present  |  2 = B1 or B2, but not A  |  3 = A plus B1 and B2

V3: ENDORSEMENT OF POLITICAL VIOLENCE
Meaning: Approval of aggressive or violent acts against political opponents.
Check for:
  A: Endorsement of violence against opponents
  B1: Derogatory labels  |  B2: Strong language or hyperbole
Scale:
  1 = None present  |  2 = B1 or B2, but not A  |  3 = A plus B1 and B2

V4: BACKING ANTI-DEMOCRATIC POLITICIANS
Meaning: Supporting candidates who undermine democratic institutions.
Check for:
  A: Support for candidates who act undemocratically
  B1: Derogatory labels  |  B2: Strong language or hyperbole
Scale:
  1 = None present  |  2 = A but not B1 or B2  |  3 = A plus B1 and B2

V5: REJECTING CROSS-PARTY COOPERATION
Meaning: Opposition to working with members of the other party.
Check for:
  A: Language that dismisses or attacks cross-party cooperation
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present

V6: GENERALISED SOCIAL DISTRUST
Meaning: Broad distrust of other people or institutions.
Check for:
  A: Language that undermines trust in others
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present

V7: SOCIAL SEGREGATION FROM OUTPARTISANS
Meaning: Desire to limit personal or social contact with the opposing party.
Check for:
  A: Language promoting separation, fear, or hostility toward opponents
  B1: Strong language or hyperbole  |  B2: References to community-damaging events
Scale:
  1 = None present  |  2 = A but not B1 or B2  |  3 = A plus B1 or B2

V8: PARTISAN INTERPRETATION OF FACTS
Meaning: Presenting or dismissing facts according to partisan viewpoint.
Check for:
  A: One-sided presentation of facts or partisan framing of a contested issue
  B: Strong language, hyperbole, or emotional appeals
Scale:
  1 = Neither present  |  2 = One present  |  3 = Both present
"""


def build_user_prompt_alt(tweet_text: str, post_type: str) -> str:
    """Alt prompt for stability analysis."""
    return f"""{CODEBOOK_ALT}
{OUTPUT_FORMAT}

Annotate this {post_type}:

\"{tweet_text}\""""


def run_stability_analysis(n: int = 30, llm_id: int = 1,
                            random_state: int = 42) -> pd.DataFrame:
    """
    Run prompt stability analysis on n tweets using two prompt variants.
    Returns a DataFrame with scores from both prompts for comparison.
    """
    subset = df.sample(n=n, random_state=random_state).reset_index(drop=True)
    model_cfg = MODELS[llm_id]
    annotate_fn = ANNOTATORS[llm_id]

    results = []
    print(f"Running stability analysis: {n} tweets × 2 prompts ({model_cfg['display_name']})...")

    for _, row in tqdm(subset.iterrows(), total=n):
        tweet_text = row['text']
        post_type  = row['post_type']

        try:
            # Canonical prompt
            ann_a = annotate_fn(tweet_text, post_type)
            time.sleep(REQUEST_DELAY)

            # Alt prompt — temporarily swap user prompt builder
            ann_b = annotate_fn.__wrapped__(tweet_text, post_type,
                                            prompt_fn=build_user_prompt_alt) \
                    if hasattr(annotate_fn, '__wrapped__') \
                    else annotate_fn(tweet_text, post_type)  # Fallback: re-run canonical
            # NOTE: For a clean implementation, pass prompt_fn as a parameter.
            # The fallback above re-runs the canonical prompt, which gives you
            # within-prompt reproducibility (also useful to report).
            time.sleep(REQUEST_DELAY)

            row_result = {"Tweet_Id": row['Tweet_Id']}
            for v in [f"V{i}" for i in range(1, 9)]:
                row_result[f"{v}_prompt_A"] = ann_a[v]['score']
                row_result[f"{v}_prompt_B"] = ann_b[v]['score']
            results.append(row_result)

        except Exception as e:
            print(f"  Skipping tweet {row['Tweet_Id']}: {e}")

    stability_df = pd.DataFrame(results)

    # Compute Spearman rho per variable
    print("\nStability Results (Spearman rho, Prompt A vs Prompt B)")
    print(f"{'Variable':<8} {'rho':>8} {'p-value':>10}")
    print("-" * 30)
    for v in [f"V{i}" for i in range(1, 9)]:
        if f"{v}_prompt_A" in stability_df.columns:
            rho, pval = spearmanr(
                stability_df[f"{v}_prompt_A"],
                stability_df[f"{v}_prompt_B"]
            )
            print(f"{v:<8} {rho:>8.3f} {pval:>10.4f}")

    return stability_df


print("Stability analysis function defined.")
print("   Run: stability_df = run_stability_analysis(n=30, llm_id=1)")
print("   This makes ~60 API calls, check cost estimate first.")

Prompt modification based on conservative/liberal leaning 

In [ ]:
# Changing system prompt to have a more conservative tone, to test stability of results across prompt variants.
SYSTEM_PROMPT_ALT_CONSERVATIVE = """You are an expert political content annotator trained in political science. 
Your task is to annotate social media posts (tweets) from the United States for anti-democratic 
attitudes and partisan animosity. You will apply a structured codebook derived from the 
Anti-democratic Attitudes and Partisan Animosity (AAPA) scale used in academic research 
(Jia et al., 2024; Voelkel et al., 2023).

You must rate each tweet on EIGHT variables (V1–V8), each on a scale of 1 to 3.
For each variable, you will provide:
  - A numeric rating (1, 2, or 3)
  - A reason citing specific language in the tweet

You MUST respond ONLY with a valid JSON object. Do not include any text before or after the JSON.
Do not include markdown code fences. Return only the raw JSON."""

# Same codebook definitions, but with more conservative wording (e.g. "negative attitudes" instead of "dislike", "endorsement" instead of "support", "aggressive or violent acts" instead of "violent tactics"). This tests whether results are sensitive to surface-level wording of the prompt, while keeping the underlying constructs the same.
CODEBOOK_ALT_CONSERVATIVE = """
CODEBOOK — Anti-Democratic Attitudes and Partisan Animosity Scale
================================================================

V1: PARTISAN ANIMOSITY
Partisan animosity is defined as "dislike for opposing partisans". 
Factors:
  A: Partisan name-calling
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V2: SUPPORT FOR UNDEMOCRATIC PRACTICES
Support for undemocratic practices is defined as "willingness to forgo democratic principles for partisan gain". 
Undemocratic practices are undemocratic tendencies or actions such as reducing polling stations in areas that support their opponents, attacking the independence of the judiciary, undermining the free press, and challenging the legitimacy of election results, and encouraging political violence. 
Factors:
  A: Shows support for undemocratic practices
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Does not satisfy A, but satisfies B1 or B2
  3 = Satisfies A, B1, and B2

V3: SUPPORT FOR PARTISAN VIOLENCE
Support for partisan violence is defined as "willingness to use violent tactics against outpartisans". 
Examples of partisan violence include sending threatening and intimidating messages to the opponent party, harassing the opponent party on the Internet, using violence in advancing their political goals or winning more races in the next election.
Factors:
  A: Shows support for partisan violence
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Does not satisfy A, but satisfies B1 or B2
  3 = Satisfies A, B1, and B2

V4: SUPPORT FOR UNDEMOCRATIC CANDIDATES
Support for undemocratic candidates is defined as "willingness to ignore undemocratic practices to elect in party candidates". 
Undemocratic candidates are oftentimes those who support the following undemocratic practices such as reducing polling stations in areas that support their opponents, attacking the independence of the judiciary, undermining the free press, and challenging the legitimacy of election results, and encouraging political violence .
Factors:
  A: Shows support for undemocratic candidates
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Satisfies A, but not B1 or B2
  3 = Satisfies A, B1, and B2

V5: OPPOSITION TO BIPARTISANSHIP
Opposition to bipartisanship is defined as "resistance to cross-partisan collaboration". 
Factors:
  A: Any name-calling or terms that reduce trust
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V6: SOCIAL DISTRUST
Social distrust is defined as "distrust of people in general". 
Factors:
  A: Any name-calling or terms that reduce trust
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V7: SOCIAL DISTANCE
Social distance is defined as "resistance to interpersonal contact with outpartisans".
Factors:
  A: Terms that increase distrust, distance, insecurity, hate, prejudice, or discrimination
  B1: Emotion or exaggeration
  B2: Events that damage communities or decrease societal trust (e.g. mass shooting)
Ratings:
  1 = Does not satisfy any factor
  2 = Satisfies A, but not B1 or B2
  3 = Satisfies A, and B1 or B2

V8: BIASED EVALUATION OF POLITICISED FACTS
Biased evaluation of politicized facts is defined as "skepticism of facts that favor the worldview of the other party". 
Factors:
  A: Partially presents political facts or discusses controversial issue with partisan stance
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist
"""

def build_user_prompt_alt(tweet_text: str, post_type: str) -> str:
    """Alt prompt for political leaning analysis."""
    return f"""{CODEBOOK_ALT_CONSERVATIVE}
{OUTPUT_FORMAT}

Annotate this {post_type}:

\"{tweet_text}\""""


def run_leaning_analysis(n: int = 30, llm_id: int = 1,
                            random_state: int = 42) -> pd.DataFrame:
    """
    Run prompt stability analysis on n tweets using two prompt variants.
    Returns a DataFrame with scores from both prompts for comparison.
    """
    subset = df.sample(n=n, random_state=random_state).reset_index(drop=True)
    model_cfg = MODELS[llm_id]
    annotate_fn = ANNOTATORS[llm_id]

    results = []
    print(f"Running stability analysis: {n} tweets × 2 prompts ({model_cfg['display_name']})...")

    for _, row in tqdm(subset.iterrows(), total=n):
        tweet_text = row['text']
        post_type  = row['post_type']

        try:
            # Canonical prompt
            ann_a = annotate_fn(tweet_text, post_type)
            time.sleep(REQUEST_DELAY)

            # Alt prompt — temporarily swap user prompt builder
            ann_b = annotate_fn.__wrapped__(tweet_text, post_type,
                                            prompt_fn=build_user_prompt_alt) \
                    if hasattr(annotate_fn, '__wrapped__') \
                    else annotate_fn(tweet_text, post_type)  # Fallback: re-run canonical
            # NOTE: For a clean implementation, pass prompt_fn as a parameter.
            # The fallback above re-runs the canonical prompt, which gives you
            # within-prompt reproducibility (also useful to report).
            time.sleep(REQUEST_DELAY)

            row_result = {"Tweet_Id": row['Tweet_Id']}
            for v in [f"V{i}" for i in range(1, 9)]:
                row_result[f"{v}_prompt_A"] = ann_a[v]['score']
                row_result[f"{v}_prompt_B"] = ann_b[v]['score']
            results.append(row_result)

        except Exception as e:
            print(f"  Skipping tweet {row['Tweet_Id']}: {e}")

    stability_df = pd.DataFrame(results)

    # Compute Spearman rho per variable
    print("\n── Stability Results (Spearman rho, Prompt A vs Prompt B) ──")
    print(f"{'Variable':<8} {'rho':>8} {'p-value':>10}")
    print("-" * 30)
    for v in [f"V{i}" for i in range(1, 9)]:
        if f"{v}_prompt_A" in stability_df.columns:
            rho, pval = spearmanr(
                stability_df[f"{v}_prompt_A"],
                stability_df[f"{v}_prompt_B"]
            )
            print(f"{v:<8} {rho:>8.3f} {pval:>10.4f}")

    return stability_df


print("Stability analysis function defined.")
print("   Run: stability_df = run_stability_analysis(n=30, llm_id=1)")
print("   This makes ~60 API calls, check cost estimate first.")
